In [7]:
import pandas as pd
test_df = pd.read_parquet("test_problems.parquet") 
contest_ids = list(test_df['contest_id'].unique())

### Download the needed generated tests first

In [ ]:
from huggingface_hub import hf_hub_download
from concurrent.futures import ThreadPoolExecutor, as_completed

def download_one(cid: int):
    try:
        path = hf_hub_download(
            repo_id="open-r1/codeforces",
            filename=f"generated_tests/test_cases_{cid:04d}.parquet",
            repo_type="dataset",
            local_dir="./hf_cache",  # optional; omit to use default cache
        )
        return cid, path
    except Exception as e:
        return cid, f"error: {e}"

with ThreadPoolExecutor(max_workers=8) as ex:
    futs = [ex.submit(download_one, cid) for cid in contest_ids]
    for fut in as_completed(futs):
        print(fut.result())

In [ ]:
from glob import glob
import pandas as pd

responses_df = pd.DataFrame()


for file in sorted(glob("/teamspace/studios/this_studio/inference/responses/SFT_unsloth_Qwen3-4B-unsloth-bnb-4bit/*/response.json")):
    responses_df = pd.concat([responses_df,pd.read_json(file)],ignore_index=True)
    
responses_df.head()

,id,index,prompt,response,retry_needed,retry_response,has_code
0,1993/A,A,You are an expert competitive programmer. You ...,"<think>\nOkay, let's see. The problem is about...",False,None,True
1,1998/A,A,You are an expert competitive programmer. You ...,"<think>\nOkay, let's tackle this problem. So, ...",True,"<think>\nOkay, let's try to figure out the cor...",False
2,1999/A,A,You are an expert competitive programmer. You ...,"<think>\nOkay, let's see. The problem is to fi...",False,None,True
3,2000/A,A,You are an expert competitive programmer. You ...,"<think>\nOkay, I need to solve this problem wh...",False,None,True
4,2001/A,A,You are an expert competitive programmer. You ...,"<think>\nOkay, let's see. We need to find the ...",False,None,True


In [31]:
from glob import glob
import pandas as pd

responses_df_2 = pd.DataFrame()


for file in sorted(glob("/teamspace/studios/this_studio/inference/responses/unsloth_Qwen3-4B-unsloth-bnb-4bit/*/response.json")):
    responses_df_2 = pd.concat([responses_df_2,pd.read_json(file)],ignore_index=True)
    
responses_df_2.head()

,id,difficulty,prompt,response,index,retry_needed,retry_response,has_code
0,1993/A,A,You are an expert competitive programmer. You ...,"<think>\nOkay, let's see. I need to find the m...",None,NaN,None,NaN
1,1998/A,A,You are an expert competitive programmer. You ...,"<think>\nOkay, I need to solve this programmin...",None,NaN,None,NaN
2,1999/A,A,You are an expert competitive programmer. You ...,"<think>\nOkay, I need to solve this problem wh...",None,NaN,None,NaN
3,2000/A,A,You are an expert competitive programmer. You ...,"<think>\nOkay, let's see. I need to determine ...",None,NaN,None,NaN
4,2001/A,None,You are an expert competitive programmer. You ...,"<think>\nOkay, let's try to figure out how to ...",A,1.0,"<think>\nOkay, I need to solve this problem wh...",0.0


In [30]:
import pandas as pd
test_df = pd.read_parquet("test_problems.parquet") 
# Subsample: top N per index
SAMPLE_PER_INDEX = {"A": 10, "B": 5, "C": 5}
test_df = pd.concat([
    test_df[test_df["index"] == k].head(n)
    for k, n in SAMPLE_PER_INDEX.items()
]).reset_index(drop=True)

In [32]:
responses_df_2.__len__()

20

In [33]:
import pandas as pd
import numpy as np

retry_mask = responses_df_2["retry_needed"].fillna(False).astype(bool)

responses_df_2["llm_response"] = np.where(
    retry_mask,
    responses_df_2["retry_response"],
    responses_df_2["response"],
)
# Merge into test_df by problem id
test_df = test_df.merge(
    responses_df_2[["id", "llm_response"]],
    on="id",
    how="left",
)

In [35]:
len(test_df)

20

In [36]:
import multiprocessing
from evaluation import *
results_df = evaluate_dataframe(test_df,n_workers=int(0.7*multiprocessing.cpu_count()))
results_df.head()

,id,aliases,contest_id,contest_name,contest_type,contest_start,contest_start_year,index,time_limit,memory_limit,title,description,input_format,output_format,interaction_format,note,examples,editorial,rating,tags,testset_size,official_tests,official_tests_complete,input_mode,generated_checker,executable,generated_tests,prompt,language,llm_response,compile_ok,error,official_passed,official_total,official_accuracy,generated_passed,generated_total,generated_accuracy,total_passed,total_total,total_accuracy,official_per_test,generated_per_test
0,1993/A,None,1993,Codeforces Round 963 (Div. 2),CF,1722782100,2024,A,1.0,256,Question Marks,Tim is doing a test consisting of $$$4n$$$ que...,The first line contains a single integer $$$t$...,"For each test case, print a single integer — t...",NaN,"In the first test case, there is exactly one q...",[{'input': '6 1 ABCD 2 AAAAAAAA 2 AAAABBBB 2 ?...,None,800.0,"[greedy, implementation]",5,[{'input': '6 1 ABCD 2 AAAAAAAA 2 AAAABB...,False,stdio,None,True,26,You are an expert competitive programmer. You ...,cpp,"<think>\nOkay, let's see. I need to find the m...",True,None,2,2,1.0,26,26,1.0,28,28,1.0,"[ok, ok]","[ok, ok, ok, ok, ok, ok, ok, ok, ok, ok, ok, o..."
1,1998/A,None,1998,Codeforces Round 965 (Div. 2),CF,1723300500,2024,A,1.0,256,Find K Distinct Points with Fixed Center,"You are given three integers $$$x_c$$$, $$$y_c...",The first line contains $$$t$$$ ($$$1 \leq t \...,"For each test case, output $$$k$$$ lines, the ...",NaN,"For the first test case, $$$\left( \frac{10}{1...","[{'input': '4 10 10 1 0 0 3 -5 -8 8 4 -5 3', '...",None,800.0,"[constructive algorithms, implementation, math]",16,[{'input': '4 10 10 1 0 0 3 -5 -8 8 4 -5 3...,False,stdio,import sys\n\ndef main():\n input_path = sy...,True,14,You are an expert competitive programmer. You ...,cpp,"<think>\nOkay, I need to solve this programmin...",True,None,15,15,1.0,14,14,1.0,29,29,1.0,"[ok, ok, ok, ok, ok, ok, ok, ok, ok, ok, ok, o...","[ok, ok, ok, ok, ok, ok, ok, ok, ok, ok, ok, o..."
2,1999/A,None,1999,Codeforces Round 964 (Div. 4),ICPC,1722954900,2024,A,1.0,256,A+B Again?,"Given a two-digit positive integer $$$n$$$, fi...",The first line contains an integer $$$t$$$ ($$...,"For each test case, output a single integer — ...",NaN,None,"[{'input': '8 77 21 40 34 19 84 10 99', 'outpu...",You can take the input as a string and output ...,800.0,"[implementation, math]",6,[{'input': '8 77 21 40 34 19 84 10 99 ...,True,stdio,None,True,0,You are an expert competitive programmer. You ...,cpp,"<think>\nOkay, I need to solve this problem wh...",True,None,6,6,1.0,0,0,0.0,6,6,1.0,"[ok, ok, ok, ok, ok, ok]",[]
3,2000/A,None,2000,Codeforces Round 966 (Div. 3),ICPC,1723560000,2024,A,1.0,256,Primary Task,Dmitry wrote down $$$t$$$ integers on the boar...,The first line of the input contains one integ...,"For each integer on the board, output ""YES"" if...",NaN,None,[{'input': '7 100 1010 101 105 2033 1019 1002'...,You need to check that the number has the form...,800.0,"[implementation, math, strings]",5,[{'input': '7 100 1010 101 105 2033 1019...,False,stdio,None,True,29,You are an expert competitive programmer. You ...,cpp,"<think>\nOkay, let's see. I need to determine ...",False,no_code_extracted,0,0,0.0,0,0,0.0,0,0,0.0,[],[]
4,2001/A,None,2001,Codeforces Round 967 (Div. 2),CF,1724164500,2024,A,1.0,256,Make All Equal,"You are given a cyclic array $$$a_1, a_2, \ldo...",Each test contains multiple test cases. The fi...,"For each test case, output a single line conta...",NaN,"In the first test case, there is only one elem...",[{'input': '7 1 1 3 1 2 3 3 1 2 2 5 5 4 3 2 1 ...,Let $$$x$$$ be one of the most frequent elemen...,800.0,"[greedy, implementation]",5,[{'input': '7 1 1 3 1 2 3 3 1 2 2 5 5 ...,False,stdio,None,True,26,You are an expert competitive programmer. You ...,cpp,"<think>\nOkay, I need to solve this problem wh...",False,no_code_extracted,0,0,0.0,0,0,0.0,0,0,0.0,[],[]


In [37]:
results_df.to_json("base_results.json",indent=4,index=False)

In [2]:
results_df = pd.read_json("SFT_results.json")
# A problem is "solved" only if it passes every test (official + generated)
# Guard against problems with 0 tests (can't pass what doesn't exist)
results_df["solved"] = (results_df["total_accuracy"] == 1.0) & (results_df["total_total"] > 0)

# ---------- Overall metrics ----------
n_total = len(results_df)
n_solved = results_df["solved"].sum()

print(f"Problems evaluated:     {n_total}")
print(f"Solved (all tests):     {n_solved} ({n_solved / n_total:.1%})")
print(f"Mean test accuracy:     {results_df['total_accuracy'].mean():.3f}")
print(f"Compile success rate:   {results_df['compile_ok'].mean():.1%}")
print()

# ---------- Per-index breakdown ----------
per_index = results_df.groupby("index").agg(
    total=("id", "count"),
    solved=("solved", "sum"),
    mean_acc=("total_accuracy", "mean"),
    compile_rate=("compile_ok", "mean"),
)
per_index["solve_rate"] = per_index["solved"] / per_index["total"]

print("Per-index breakdown:")
print(per_index[["total", "solved", "solve_rate", "mean_acc", "compile_rate"]]
      .round(3).to_string())

Problems evaluated:     20
Solved (all tests):     3 (15.0%)
Mean test accuracy:     0.188
Compile success rate:   75.0%

Per-index breakdown:
       total  solved  solve_rate  mean_acc  compile_rate
index                                                   
A         10       3         0.3     0.300           0.7
B          5       0         0.0     0.093           0.8
C          5       0         0.0     0.057           0.8


In [38]:
results_df = pd.read_json("base_results.json")
# A problem is "solved" only if it passes every test (official + generated)
# Guard against problems with 0 tests (can't pass what doesn't exist)
results_df["solved"] = (results_df["total_accuracy"] == 1.0) & (results_df["total_total"] > 0)

# ---------- Overall metrics ----------
n_total = len(results_df)
n_solved = results_df["solved"].sum()

print(f"Problems evaluated:     {n_total}")
print(f"Solved (all tests):     {n_solved} ({n_solved / n_total:.1%})")
print(f"Mean test accuracy:     {results_df['total_accuracy'].mean():.3f}")
print(f"Compile success rate:   {results_df['compile_ok'].mean():.1%}")
print()

# ---------- Per-index breakdown ----------
per_index = results_df.groupby("index").agg(
    total=("id", "count"),
    solved=("solved", "sum"),
    mean_acc=("total_accuracy", "mean"),
    compile_rate=("compile_ok", "mean"),
)
per_index["solve_rate"] = per_index["solved"] / per_index["total"]

print("Per-index breakdown:")
print(per_index[["total", "solved", "solve_rate", "mean_acc", "compile_rate"]]
      .round(3).to_string())

Problems evaluated:     20
Solved (all tests):     8 (40.0%)
Mean test accuracy:     0.414
Compile success rate:   70.0%

Per-index breakdown:
       total  solved  solve_rate  mean_acc  compile_rate
index                                                   
A         10       5         0.5     0.525           0.8
B          5       2         0.4     0.400           0.6
C          5       1         0.2     0.207           0.6
